In [3]:
# ============================================
# TASK 2.1 — ALSO-X / sample-based MILP
# FCR-D Up reserve bid under P90 requirement
# ============================================

import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

load_in = np.load("../data/load_in.npy")
load_out = np.load("../data/load_out.npy")
F_up_in = np.load("../data/F_up_in.npy")
F_up_out = np.load("../data/F_up_out.npy")

N_in, M = F_up_in.shape
N_out, _ = F_up_out.shape

epsilon = 0.10          # P90 requirement -> max 10% violation
M_big = 10000           # Big-M
q = int(epsilon * N_in * M)

model_alsox = gp.Model("Task_2_1_ALSOX_FCRD_Up")
model_alsox.setParam("OutputFlag", 0)

# Decision variable: reserve capacity bid [kW]
c_up = model_alsox.addVar(lb=0, name="c_up")

# Binary violation variable
y = model_alsox.addVars(N_in, M, vtype=GRB.BINARY, name="y")

# If c_up > available flexibility, violation is allowed through y
for s in range(N_in):
    for m in range(M):
        model_alsox.addConstr(
            c_up - F_up_in[s, m] <= M_big * y[s, m],
            name=f"availability_{s}_{m}"
        )

# Violation budget: at most 10% of all sample-minute cases
model_alsox.addConstr(
    gp.quicksum(y[s, m] for s in range(N_in) for m in range(M)) <= q,
    name="P90_violation_budget"
)

# Objective: maximize reserve bid
model_alsox.setObjective(c_up, GRB.MAXIMIZE)

model_alsox.optimize()

if model_alsox.status == GRB.OPTIMAL:
    c_up_alsox = c_up.X
    print("========== ALSO-X / MILP RESULT ==========")
    print(f"Optimal FCR-D Up reserve bid: {c_up_alsox:.2f} kW")
    print(f"Allowed violations: {q} out of {N_in*M}")
    print(f"In-sample violation count: {sum(y[s,m].X for s in range(N_in) for m in range(M)):.0f}")
else:
    print("ALSO-X model not optimal. Status:", model_alsox.status)

GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
# ============================================
# TASK 2.1 — CVaR approximation
# FCR-D Up reserve bid under P90 requirement
# ============================================

model_cvar = gp.Model("Task_2_1_CVaR_FCRD_Up")
model_cvar.setParam("OutputFlag", 0)

# Decision variable: reserve capacity bid [kW]
c_up_cvar = model_cvar.addVar(lb=0, name="c_up")

# CVaR variables
beta = model_cvar.addVar(lb=0, name="beta")          # VaR-like threshold
shortfall = model_cvar.addVars(N_in, M, lb=0, name="shortfall")

# Shortfall constraints
for s in range(N_in):
    for m in range(M):
        model_cvar.addConstr(
            c_up_cvar - F_up_in[s, m] <= shortfall[s, m],
            name=f"shortfall_{s}_{m}"
        )

        model_cvar.addConstr(
            beta <= shortfall[s, m],
            name=f"beta_bound_{s}_{m}"
        )

# CVaR conservative approximation
model_cvar.addConstr(
    gp.quicksum(shortfall[s, m] for s in range(N_in) for m in range(M)) / (N_in * M)
    <= (1 - epsilon) * beta,
    name="CVaR_constraint"
)

# Objective: maximize reserve bid
model_cvar.setObjective(c_up_cvar, GRB.MAXIMIZE)

model_cvar.optimize()

if model_cvar.status == GRB.OPTIMAL:
    c_up_cvar_val = c_up_cvar.X
    print("\n========== CVaR RESULT ==========")
    print(f"Optimal FCR-D Up reserve bid: {c_up_cvar_val:.2f} kW")
    print(f"beta value: {beta.X:.2f}")
else:
    print("CVaR model not optimal. Status:", model_cvar.status)

GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
# ============================================
# Task 2.1 Summary
# ============================================

task_2_1_results = pd.DataFrame({
    "Method": ["ALSO-X / MILP", "CVaR approximation"],
    "Optimal reserve bid [kW]": [c_up_alsox, c_up_cvar_val]
})

task_2_1_results